<a href="https://colab.research.google.com/github/Kaunaingul-ai/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kaunaingul-ai/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked action queue

I convert the model's decline scores into a ranked review queue rather than an automatic refresh decision.

Each recommended page receives simple reason codes based on observable, public-safe signals. These reason codes are intended to help an editor understand why a page surfaced in the queue; they are not explanations of causality.

The highest-ranked pages are marked `REVIEW_FIRST`. A human editor should then consider the page's purpose, seasonality, query intent, content quality, and business priorities before deciding whether any change is appropriate.

In [6]:
import os
import subprocess
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier


# ------------------------------------------------
# Load the public-safe starter data
# ------------------------------------------------
REPO_URL = "https://github.com/Kaunaingul-ai/flyrank-ml-internship"
REPO_DIR = "/content/flyrank-ml-internship"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

data_path = os.path.join(
    REPO_DIR,
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_path)


# ------------------------------------------------
# Final feature set
# ------------------------------------------------
feature_cols = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "word_count"
]

X = df[feature_cols].copy()
y = (df["trend_direction"] == "down").astype(int)


# ------------------------------------------------
# Same Week-5 style validation split
# ------------------------------------------------
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)

scores = model.predict_proba(X_valid)[:, 1]


# ------------------------------------------------
# Build a public-safe ranked queue
# ------------------------------------------------
queue = df.loc[X_valid.index, [
    "content_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
]].copy()

queue["decline_score"] = scores
queue["actual_decline"] = y_valid.values


# ------------------------------------------------
# Human-readable reason codes
# ------------------------------------------------
def reason_codes(row):
    reasons = []

    if row["days_since_last_update"] > 90:
        reasons.append("STALE")

    if row["ctr"] <= 0.10:
        reasons.append("LOW_CTR")

    if row["avg_position"] > 20:
        reasons.append("WEAK_POSITION")

    if row["impressions_90d"] >= 5000:
        reasons.append("HIGH_VISIBILITY")

    if not reasons:
        reasons.append("MODEL_PATTERN")

    return " | ".join(reasons)


queue["reason_codes"] = queue.apply(reason_codes, axis=1)

queue = queue.sort_values(
    "decline_score",
    ascending=False
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

queue["recommended_action"] = np.where(
    queue["rank"] <= 50,
    "REVIEW_FIRST",
    "MONITOR"
)


# ------------------------------------------------
# Display top recommendations
# ------------------------------------------------
top20 = queue.head(20)[[
    "rank",
    "content_id",
    "decline_score",
    "reason_codes",
    "recommended_action"
]]

print("TOP 20 RANKED ACTIONS")
print(top20.round({"decline_score": 3}).to_string(index=False))

print("\nTop-50 summary:")
print("Pages marked REVIEW_FIRST:",
      (queue["recommended_action"] == "REVIEW_FIRST").sum())

print(
    "Observed declining pages among top 50:",
    int(queue.head(50)["actual_decline"].sum())
)

TOP 20 RANKED ACTIONS
 rank           content_id  decline_score                    reason_codes recommended_action
    1 content_25a763874cf0          0.839           STALE | WEAK_POSITION       REVIEW_FIRST
    2 content_d0e81e632d6f          0.839 STALE | LOW_CTR | WEAK_POSITION       REVIEW_FIRST
    3 content_d06315b2b326          0.838                 STALE | LOW_CTR       REVIEW_FIRST
    4 content_cae546aab87e          0.837 STALE | LOW_CTR | WEAK_POSITION       REVIEW_FIRST
    5 content_f192f3938827          0.836 STALE | LOW_CTR | WEAK_POSITION       REVIEW_FIRST
    6 content_65bf969d1247          0.836                 STALE | LOW_CTR       REVIEW_FIRST
    7 content_d3b6b4aae679          0.836           STALE | WEAK_POSITION       REVIEW_FIRST
    8 content_b7c3994aa0e8          0.835 STALE | LOW_CTR | WEAK_POSITION       REVIEW_FIRST
    9 content_ffef3141126f          0.834                 STALE | LOW_CTR       REVIEW_FIRST
   10 content_d829395fda44          0.834       

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use and limits

This playbook is intended for content editors, SEO analysts, or content operations teams who need to prioritize a large set of pages for review when time is limited.

The ranked queue should be used as a decision-support tool. A high decline score means that a page resembles patterns associated with decline in the observed data; it does not prove that the page is declining because of any specific feature, nor does it prove that refreshing the page will improve search performance.

The recommendations are most appropriate for pages similar to the public-safe starter dataset used to build the model. They should not be treated as universally valid across different clients, industries, search environments, or future time periods without new validation.

Human review remains necessary because the model does not observe all relevant context, including seasonality, page purpose, search intent, content quality, editorial strategy, or business priorities.

In [7]:
playbook_limits = {
    "decision_support_only": True,
    "claims_causality": False,
    "automatic_refresh_decision": False,
    "requires_human_review": True,
    "validated_for_all_clients": False
}

for item, value in playbook_limits.items():
    print(f"{item}: {value}")


decision_support_only: True
claims_causality: False
automatic_refresh_decision: False
requires_human_review: True
validated_for_all_clients: False


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review and no-go list

Before acting on any ranked recommendation, a human reviewer should inspect the page in context. Important checks include page purpose, seasonality, current search intent, content quality, business importance, recent editorial changes, and whether the page is already being updated through another workflow.

The model should never automatically delete, rewrite, merge, redirect, unpublish, or refresh a page solely because of its decline score. It should also never be used to claim that a particular feature caused the decline.

The safe workflow is therefore: rank pages → review the evidence → inspect the page manually → choose an editorial action only when the context supports it.

In [8]:
human_review_checks = [
    "page_purpose",
    "seasonality",
    "search_intent",
    "content_quality",
    "business_priority",
    "recent_editorial_changes"
]

never_automate = [
    "delete_page",
    "unpublish_page",
    "rewrite_page",
    "merge_page",
    "redirect_page",
    "claim_causality"
]

print("HUMAN REVIEW CHECKS")
for item in human_review_checks:
    print("-", item)

print("\nNO-GO AUTOMATION LIST")
for item in never_automate:
    print("-", item)

print("\nHuman review required:", True)
print("Automatic content action allowed:", False)


HUMAN REVIEW CHECKS
- page_purpose
- seasonality
- search_intent
- content_quality
- business_priority
- recent_editorial_changes

NO-GO AUTOMATION LIST
- delete_page
- unpublish_page
- rewrite_page
- merge_page
- redirect_page
- claim_causality

Human review required: True
Automatic content action allowed: False


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and retraining triggers

The recommendation system should be monitored after deployment because page behavior and search conditions can change over time.

Retraining or revalidation should be considered if model performance drops materially, if the distribution of important features shifts, if the share of declining pages changes substantially, or if the model begins producing recommendation patterns that differ strongly from the original validation data.

A practical trigger would be a clear decline in ranking quality such as a lower Precision@50 on newly labeled data, especially if the drop persists across multiple review cycles.

In [9]:
monitoring_triggers = {
    "precision_at_50_drop": True,
    "feature_distribution_shift": True,
    "label_distribution_shift": True,
    "new_client_mix": True,
    "search_environment_change": True
}

retrain_if = [
    "Precision@50 drops materially on new labeled data",
    "Important feature distributions shift substantially",
    "Decline-rate prevalence changes materially",
    "New client groups differ from the original training population",
    "Recommendation quality becomes inconsistent across review cycles"
]

print("MONITORING SIGNALS")
for item, value in monitoring_triggers.items():
    print(f"- {item}: {value}")

print("\nRETRAIN / REVALIDATE IF")
for item in retrain_if:
    print("-", item)

MONITORING SIGNALS
- precision_at_50_drop: True
- feature_distribution_shift: True
- label_distribution_shift: True
- new_client_mix: True
- search_environment_change: True

RETRAIN / REVALIDATE IF
- Precision@50 drops materially on new labeled data
- Important feature distributions shift substantially
- Decline-rate prevalence changes materially
- New client groups differ from the original training population
- Recommendation quality becomes inconsistent across review cycles


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for the paper

I export the ranked recommendation queue and a compact top-50 review table so the deployed paper can reuse the same evidence generated in this notebook.

Only anonymized, public-safe fields are written to the output files. No client names, domains, URLs, private queries, or raw private identifiers are exported.

In [10]:
output_dir = os.path.join(REPO_DIR, "work", "outputs")
os.makedirs(output_dir, exist_ok=True)

# Full public-safe recommendation queue
queue_export = queue[[
    "rank",
    "content_id",
    "decline_score",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update",
    "reason_codes",
    "recommended_action"
]].copy()

# Compact top-50 table for the paper
top50_export = queue_export.head(50).copy()

queue_path = os.path.join(
    output_dir,
    "w07_ranked_action_queue.csv"
)

top50_path = os.path.join(
    output_dir,
    "w07_top50_review_first.csv"
)

queue_export.to_csv(queue_path, index=False)
top50_export.to_csv(top50_path, index=False)

print("EXPORT COMPLETE")
print("Full queue:", queue_path)
print("Top-50 table:", top50_path)

print("\nRows exported:")
print("Full queue:", len(queue_export))
print("Top-50 table:", len(top50_export))

print("\nPublic-safe columns:")
print(queue_export.columns.tolist())

EXPORT COMPLETE
Full queue: /content/flyrank-ml-internship/work/outputs/w07_ranked_action_queue.csv
Top-50 table: /content/flyrank-ml-internship/work/outputs/w07_top50_review_first.csv

Rows exported:
Full queue: 6000
Top-50 table: 50

Public-safe columns:
['rank', 'content_id', 'decline_score', 'impressions_90d', 'ctr', 'avg_position', 'content_age_days', 'days_since_last_update', 'reason_codes', 'recommended_action']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.